# Test Pipeline

Small notebook for testing the extracted line-mode pipeline on one SWORD reach selection.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np

from main_pipeline import extract_line_modes_auto
from supporting_geometry import merge_paths_mainstem_only
from supporting_plotting import plot_modes_plotly

In [ ]:
GPKG_PATH = Path("/Volumes/PhD/SWORD/v17b/GPKG/sa_sword_reaches_v17b.gpkg")
TARGET_CRS = "EPSG:3857"

if not GPKG_PATH.exists():
    raise FileNotFoundError(f"Update GPKG_PATH; file not found: {GPKG_PATH}")

df = gpd.read_file(GPKG_PATH)
df.shape, df.crs

In [ ]:
reach = (
    df[(df["path_segs"].isin([342, 343, 344])) & (df["river_name"] == "Purus River")]
    .to_crs(TARGET_CRS)
)

reach[["reach_id", "river_name", "path_segs", "width", "geometry"]].head()

In [ ]:
merged = merge_paths_mainstem_only(reach)
width_m = float(reach["width"].median())

print("reach rows:", len(reach))
print("median width:", round(width_m, 2))
print("merged geometry:", merged.geom_type)
print("merged length:", round(merged.length, 2))

In [ ]:
result = extract_line_modes_auto(
    merged,
    width_m=width_m,
    make_plots=True,
    prune_dist_abs=width_m,
)

print("step_m:", result["step_m"])
print("boundary sigmas:", np.round(result["boundary_sigmas"], 2))
print("mode sigmas:", np.round(result["mode_sigmas"], 2))
print("mid:", result["mid_info"])
print("terminal:", result["terminal_info"])

for sigma, label in zip(result["mode_sigmas"], result["mode_labels"]):
    print(label, round(float(sigma), 2))

In [ ]:
plot_labels = [
    f"{label} | sigma={sigma:.0f}m | sc={sc}"
    for sigma, label, sc in zip(
        result["mode_sigmas"],
        result["mode_labels"],
        result["curvature_sign_changes"],
    )
]

fig = plot_modes_plotly(result["ls_equal"], result["modes"], labels=plot_labels, show=False)
fig